In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install Libraries untuk akselerasi LLM lokal
!pip install -q transformers bitsandbytes accelerate tqdm

import os
import json
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("✅ Cell 1 Selesai: Drive berhasil di-mount dan library siap digunakan!")

In [ ]:
from huggingface_hub import login
login("[INSERT YOUR HF_TOKEN]")

In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Konfigurasi BitsAndBytes untuk hemat VRAM T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("⏳ Memuat tokenizer dan model Llama 3 4-bit (Tunggu sekitar 3-5 menit)...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Cell 2 Selesai: Llama 3 8B 4-Bit berhasil dimuat ke dalam GPU T4!")

In [ ]:
import json
import os

def load_character_spines(marcel_path, jessica_path):
    """
    Membaca profile asli langsung dari file JSON tanpa hardcode text.
    """
    with open(marcel_path, 'r') as f:
        marcel_spine = json.load(f)
    with open(jessica_path, 'r') as f:
        jessica_spine = json.load(f)
    return marcel_spine, jessica_spine

def build_system_prompt(character_name, spine_data, current_intensity):
    """
    Merakit system prompt secara dinamis mengekstrak dari isi file JSON asli.
    """
    if character_name == "Marcel":
        core = spine_data["identity_core"]
        style = spine_data["linguistic_style"]
        ego = spine_data["ego_profile"]
        cognitive = spine_data["cognitive_style"]

        prompt = f"""Kamu adalah Marcel. Kepribadianmu: {core['self_summary']}.
Motto hidupmu: {core['motto']}. Ketakutan terdalammu: {core['deepest_fear']}.
Gaya Kognitif & Berdebat: {cognitive['debate_style']}. Jika kamu salah, prinsipmu: {ego['signature_behavior']}.
Gaya Menulis Chat: {style['tone_default']}, panjang pesan {style['message_length']}, menggunakan aturan {style['capitalization']}, dan tanda baca {style['punctuation']}.
Penanda Autentisitas: {style['authenticity_markers']}.
Jika situasi sedang mendingin/menjauh, gunakan pola Cold Mode: {style['cold_mode_markers']}.
"""
    else: # Jessica
        traits = spine_data["identity_traits"]
        style = spine_data["linguistic_style"]
        rules = spine_data["social_dynamics_rules"]

        prompt = f"""Kamu adalah Jessica. Energi utamamu: {traits['core_energy']}.
Gaya Sosial: {traits['social_style']}. Pola Inisiatif: {traits['initiative_pattern']}.
Gaya Menulis Chat: {style['tone_default']}, panjang pesan {style['message_length']}, menggunakan aturan {style['capitalization']}, dan tanda baca {style['punctuation']}.
Jika kamu sedang bersemangat (Excited Mode), tandanya: {style['excited_markers']}.
Jika situasi sedang berkonflik/marah (Cold Mode), responmu mengikuti aturan: {rules['conflict_response']['if_upset_at_you']} dan {style['cold_mode_markers']}.
"""

    prompt += f"\nINTENSITAS EMOSI SAAT INI (Ditentukan Sutradara): {current_intensity}\n"
    prompt += "Tugasmu: Balas chat terakhir dengan sangat natural, singkat (seperti WhatsApp), menjiwai profil di atas, dan JANGAN mengetikkan nama karakter di awal teks balasan."
    return prompt

def generate_chat_batch_v2(director_instruction, intensity, chat_history_window, current_turn, marcel_spine, jessica_spine, batch_size=16):
    """
    Eksekusi paralel batched generation menggunakan data dari JSON Spine.
    """
    spine = marcel_spine if current_turn == "Marcel" else jessica_spine
    system_prompt = build_system_prompt(current_turn, spine, intensity)

    history_str = "\n".join([f"{c['sender']}: {c['text']}" for c in chat_history_window])

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}

ARAHAN KONTEKS DARI SUTRADARA:
{director_instruction}
<|eot_id|><|start_header_id|>user<|end_header_id|>
Berikut adalah riwayat obrolan terakhir:
{history_str}

Berikan respons chat pendek selanjutnya sebagai {current_turn}:
<|start_header_id|>assistant<|end_header_id|>"""

    prompts = [prompt] * batch_size
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=True,
            temperature=0.75,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    responses = []
    input_length = inputs.input_ids.shape[1]
    for output in outputs:
        generated_text = tokenizer.decode(output[input_length:], skip_special_tokens=True).strip()
        # Clean up accidental prefixes
        if generated_text.lower().startswith(f"{current_turn.lower()}:"):
            generated_text = generated_text[len(current_turn)+1:].strip()
        responses.append(generated_text)

    return responses

print("✅ Cell 3 Selesai: Engine Loader berbasis JSON Berhasil Disiapkan!")

In [ ]:
# INSERT THE PLACEHOLDER FOR THE SAVE DIRECTORY
SAVE_DIR = "[]"
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# INSERT AND CHANGE THE PLACEHOLDER
MARCEL_JSON_PATH = "[]"
JESSICA_JSON_PATH = "[]
def run_year_session_v2():
    print("=== 🎬 INTERACTIVE SESSION CONTROLLER (JSON-BASED) ===")

    # Load Spine Data secara realtime
    if not os.path.exists(MARCEL_JSON_PATH) or not os.path.exists(JESSICA_JSON_PATH):
        print("❌ ERROR: File JSON Spine tidak ditemukan di path Drive yang ditentukan. Cek kembali penempatan file lu!")
        return

    marcel_spine, jessica_spine = load_character_spines(MARCEL_JSON_PATH, JESSICA_JSON_PATH)
    print("🎯 Spine data untuk Marcel & Jessica berhasil dimuat dari Google Drive!")

    current_year = input("▶️ Masukkan Nama Sesi / Tahun (Contoh: Tahun 1): ").strip()
    target_chats = int(input("📊 Masukkan Target Jumlah Baris Chat untuk Sesi Ini: "))
    intensity = input("🔥 Set Tingkat Intensitas Emosi Sesi Ini (Contoh: Low flirty / High conflict / Cold detachment): ").strip()

    print("\n--- 📝 INPUT PLOT CONTEXT DARI DIRECTOR ---")
    director_instruction = input("Ketik skenario/pemicu masalah untuk sesi ini:\n> ")

    file_path = os.path.join(SAVE_DIR, f"data_simulasi_{current_year.lower().replace(' ', '_')}.json")

    session_data = []
    chat_history_window = []

    # Logic Resume Otomatis dari Drive
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            session_data = json.load(f)
        print(f"🔄 Checkpoint ditemukan! Melanjutkan sesi {current_year} dari baris ke-{len(session_data)}")
        chat_history_window = session_data[-10:] if len(session_data) >= 10 else session_data
    else:
        # Jika benar-benar baru, berikan pemicu awal dari template entry point Jessica di JSON
        initial_opener = jessica_spine["intent_templates"]["re_engagement_opener"]
        chat_entry = {"index": 1, "year": current_year, "sender": "Jessica", "text": initial_opener}
        session_data.append(chat_entry)
        chat_history_window.append(chat_entry)

    batch_size = 4
    current_turn = "Marcel" if session_data[-1]["sender"] == "Jessica" else "Jessica"

    pbar = tqdm(initial=len(session_data), total=target_chats, desc=f"Processing {current_year}")

    while len(session_data) < target_chats:
        # Panggil Generator dengan parameter dinamis dari JSON Spine
        responses = generate_chat_batch_v2(
            director_instruction, intensity, chat_history_window, current_turn,
            marcel_spine, jessica_spine, batch_size=batch_size
        )

        for resp in responses:
            if len(session_data) >= target_chats:
                break

            chat_entry = {
                "index": len(session_data) + 1,
                "year": current_year,
                "sender": current_turn,
                "text": resp
            }

            session_data.append(chat_entry)
            chat_history_window.append(chat_entry)

            if len(chat_history_window) > 10:
                chat_history_window.pop(0)

            current_turn = "Jessica" if current_turn == "Marcel" else "Marcel"
            pbar.update(1)

        # Autosave berkala per 512 baris
        if len(session_data) % 512 == 0:
            with open(file_path, "w") as f:
                json.dump(session_data, f, indent=4)

    # Final Save Sesi
    with open(file_path, "w") as f:
        json.dump(session_data, f, indent=4)

    print(f"\n✨ SUCCESS! Sesi {current_year} done with total of {len(session_data)} data.")
    print("Waiting for another input")

# Jalankan Sesi
run_year_session_v2()

In [ ]:
import json

# PATH FILE FOR WHERE YOU SAVE THE DATA
file_path = '[]'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Sneak peek up to 10 chats
    print("=== SNEAK PEEK DATA SIMULASI ===")
    print(json.dumps(data[:10], indent=2, ensure_ascii=False))
    print(f"\n HARVESTED TOTAL DATA {len(data)} Line.")

except FileNotFoundError:
    print(f"Eror: No file in '{file_path}'.")
except Exception as e:
    print(f"Another error on the file: {e}")